# Rubric Output Explorer

This notebook loads article analysis JSON files from `analysis_outputs/` into a pandas DataFrame so you can inspect, summarize, and filter the rubric outputs.

Note: `global_instructions` lives in `rubric.md` as rubric metadata, so it is not a per-article output column. The filterable columns here are the actual result fields such as `article_relevance`, `article_type`, `source_visibility_balance`, and the other enum or boolean outputs.

In [ ]:
# Run once if needed:
# %pip install pandas matplotlib jupyterlab

In [ ]:
from pathlib import Path
from html import escape
import json

import pandas as pd
from IPython.display import HTML, display

from article_analysis.rubric import load_rubric

RESULTS_DIR = Path("analysis_outputs")
RUBRIC_PATH = Path("rubric.md")

In [ ]:
rubric = load_rubric(RUBRIC_PATH)
rubric_fields = rubric["fields"]

evidence_fields = [name for name in rubric_fields if name.endswith("_evidence_snippets") or name == "evidence_snippets"]
enum_fields = [name for name, config in rubric_fields.items() if "enum" in config]
boolean_fields = [name for name, config in rubric_fields.items() if config.get("type") == "boolean"]
filterable_fields = enum_fields + boolean_fields
if "confidence" in rubric_fields:
    filterable_fields.append("confidence")

filterable_fields

In [ ]:
def load_analysis_results(results_dir: Path = RESULTS_DIR) -> pd.DataFrame:
    rows = []
    for path in sorted(results_dir.glob("*.json")):
        data = json.loads(path.read_text(encoding="utf-8"))
        row = {
            "article_id": path.stem,
            "result_file": path.name,
        }
        row.update(data)
        rows.append(row)

    if not rows:
        raise ValueError(f"No JSON files found in {results_dir}")

    frame = pd.DataFrame(rows)
    return frame.sort_values("article_id").reset_index(drop=True)


def apply_filters(frame: pd.DataFrame, **filters) -> pd.DataFrame:
    filtered = frame.copy()
    for column, value in filters.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple, set)):
            filtered = filtered[filtered[column].isin(list(value))]
        else:
            filtered = filtered[filtered[column] == value]
    return filtered.reset_index(drop=True)


def show_value_counts(frame: pd.DataFrame, columns=None) -> pd.DataFrame:
    columns = columns or filterable_fields
    tables = []
    for column in columns:
        counts = frame[column].value_counts(dropna=False).rename_axis("value").reset_index(name="count")
        counts.insert(0, "field", column)
        tables.append(counts)
    return pd.concat(tables, ignore_index=True)


def _foldable_snippet_html(value, label: str = "snippets") -> str:
    if isinstance(value, list):
        snippets = [str(item) for item in value if str(item).strip()]
    elif pd.isna(value):
        snippets = []
    else:
        snippets = [str(value)]

    if not snippets:
        return ""

    items = "".join(f"<li>{escape(snippet)}</li>" for snippet in snippets)
    return (
        f"<details><summary>{escape(label)} ({len(snippets)})</summary>"
        f"<ul>{items}</ul></details>"
    )


def display_results_with_evidence(
    frame: pd.DataFrame,
    columns=None,
    snippet_columns=None,
):
    columns = columns or [column for column in frame.columns if column not in evidence_fields]
    snippet_columns = snippet_columns or [column for column in evidence_fields if column in frame.columns]
    table = frame[columns + snippet_columns].copy()

    for column in snippet_columns:
        label = column.replace("_evidence_snippets", "")
        table[column] = table[column].apply(lambda value: _foldable_snippet_html(value, label=label))

    formatters = {column: lambda value: value for column in snippet_columns}
    styler = table.style.format(formatters, escape=False)
    display(HTML(styler.to_html()))


In [ ]:
df = load_analysis_results()
display_results_with_evidence(
    df,
    columns=[
        "article_id",
        "article_relevance",
        "article_type",
        "palestinian_civilian_status_representation",
        "dehumanization_present",
        "dehumanization_forms",
        "incitement_or_advocacy_of_crimes",
        "discrediting_palestinian_journalists",
        "palestinian_voice_presence",
        "israeli_voice_presence",
        "confidence",
    ],
    snippet_columns=[
        "palestinian_civilian_status_representation_evidence_snippets",
        "dehumanization_present_evidence_snippets",
        "dehumanization_forms_evidence_snippets",
        "incitement_or_advocacy_of_crimes_evidence_snippets",
        "discrediting_palestinian_journalists_evidence_snippets",
        "palestinian_voice_presence_evidence_snippets",
        "israeli_voice_presence_evidence_snippets",
        "evidence_snippets",
    ],
)

df

In [ ]:
for column in filterable_fields:
    values = df[column].dropna().tolist()
    unique_values = sorted({str(value) for value in values})
    print(f"{column}: {unique_values}")

In [ ]:
summary = show_value_counts(df)
summary

In [ ]:
# Example: filter to high-relevance analysis pieces.
filtered = apply_filters(
    df,
    article_relevance="high",
    article_type="analysis",
)

filtered[[
    "article_id",
    "article_relevance",
    "article_type",
    "source_visibility_balance",
    "source_visibility_skew_direction",
    "palestinian_voice_presence",
    "israeli_voice_presence",
    "confidence",
]]

In [ ]:
# Change these values as needed.
custom_filtered = apply_filters(
    df,
    article_relevance="high",
    source_visibility_balance="heavily_skewed",
    dehumanization_present=False,
)

display_results_with_evidence(
    custom_filtered,
    columns=[
        "article_id",
        "article_relevance",
        "source_visibility_balance",
        "dehumanization_present",
        "palestinian_voice_presence",
        "israeli_voice_presence",
        "confidence",
    ],
    snippet_columns=[
        "dehumanization_present_evidence_snippets",
        "palestinian_voice_presence_evidence_snippets",
        "israeli_voice_presence_evidence_snippets",
        "evidence_snippets",
    ],
)

custom_filtered